## project Leonardo Crespi 908918

## Use of Generative AI
Generative AI was used to debug and refine code in this project, particularly in Point 3 for assigning labels to the 'gpa_change' trend. Having learned the method, I then applied it independently in Point 6

The dataset student_dropout_risk_2026.csv.gz contains one row for each university student (identified by person_id), with demographic, economic, academic and behavioural information. The column dropout is 1 if the student has dropped out, 0 otherwise. All columns whose values are 0/1 are boolean flags

In [1]:
import pandas as pd

In [2]:
sdr = pd.read_csv('student_dropout_risk_2026.csv')
sdr

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,library_visits_per_month,lms_logins_per_week,extracurricular_activities,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout
0,1,23,Male,Brazil,0,0,Low,1,0,1,...,4,16.3,0,3,8,0,6,37,5.7,0
1,2,18,Other,USA,1,1,High,1,1,0,...,3,8.4,1,1,7,0,6,15,5.4,1
2,3,24,Other,USA,0,0,High,0,1,1,...,3,11.3,1,0,2,0,3,4,4.3,0
3,4,25,Female,USA,0,1,Middle,1,1,0,...,2,13.2,1,1,5,0,7,10,6.7,0
4,5,15,Female,Canada,0,0,Low,1,1,0,...,4,7.4,1,0,7,0,10,24,5.8,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,99996,25,Female,USA,0,0,Middle,1,1,1,...,10,6.2,0,0,4,0,6,11,5.9,0
99996,99997,24,Female,India,0,1,Middle,0,1,1,...,0,8.7,1,3,7,1,5,21,5.5,1
99997,99998,27,Female,USA,0,0,Middle,1,1,1,...,8,7.5,0,2,2,0,6,29,5.8,0
99998,99999,21,Male,UK,0,0,High,0,1,0,...,6,13.6,1,2,4,1,10,43,6.2,0


1 Extract all students that study Online, have an attendance_rate smaller than 50, and have more than 5 missed_deadlines. Store them in a dataframe called at_risk

In [3]:
at_risk = sdr[(sdr['study_mode'] == 'Online') & (sdr['attendance_rate'] < 50) & (sdr['missed_deadlines'] > 5)]
at_risk

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,library_visits_per_month,lms_logins_per_week,extracurricular_activities,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout
478,479,22,Male,Germany,0,1,High,0,0,1,...,0,13.3,1,8,8,1,3,30,6.1,1
581,582,17,Female,India,1,0,Low,1,1,1,...,3,12.4,1,11,6,0,8,0,7.1,1
642,643,29,Male,Mexico,0,0,Low,1,1,1,...,4,14.3,0,9,10,0,7,9,4.0,1
1216,1217,15,Male,UK,0,0,Low,1,1,1,...,2,6.6,0,7,7,0,5,11,5.4,1
1316,1317,21,Male,Brazil,0,0,Low,0,1,1,...,3,7.8,0,17,5,0,4,6,8.1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
96590,96591,23,Female,USA,1,0,Low,1,1,1,...,4,9.6,0,10,8,0,8,0,7.9,0
96648,96649,26,Female,USA,1,0,Low,1,1,1,...,2,12.4,0,8,9,1,6,10,6.5,1
97775,97776,19,Male,Germany,0,0,Low,1,0,1,...,6,7.5,1,8,6,1,10,19,8.8,1
99696,99697,26,Female,USA,1,0,Middle,1,0,1,...,2,14.3,1,16,9,1,8,12,7.8,1


2 Then compute, for each course_type, how many of those students there are and what fraction of them has dropped out. Store the result in a dataframe called at_risk_by_course, with columns students and dropout_rate, sorted by decreasing dropout_rate

In [4]:
at_risk_pe = at_risk.groupby('course_type')['person_id'].count().reset_index()
at_risk_pe.columns = ['course_type', 'students']

at_risk_dr = at_risk.groupby('course_type')['dropout'].mean().reset_index()
at_risk_dr.columns = ['course_type', 'dropout_rate']

at_risk_by_course = pd.merge(at_risk_pe, at_risk_dr, on='course_type')
at_risk_by_course = at_risk_by_course.sort_values('dropout_rate', ascending=False)
at_risk_by_course

,course_type,students,dropout_rate
3,Law & Social Sciences,29,0.896552
4,STEM,47,0.765957
1,Business,47,0.765957
0,Arts & Humanities,40,0.750000
2,Health Sciences,34,0.647059


3 Add to the dataframe the following columns:

pass_rate: the ratio between units_passed and units_enrolled;

gpa_change: the difference between gpa_semester_2 and gpa_semester_1;

gpa_trend: a string that is "improving" if gpa_change is larger than 0.5, "declining" if gpa_change is smaller than -0.5, and "stable" otherwise

In [5]:
sdr['pass_rate'] = sdr['units_passed'] / sdr['units_enrolled']
sdr['gpa_change'] = sdr['gpa_semester_2'] - sdr['gpa_semester_1']
sdr['gpa_trend'] = 'stable'
sdr.loc[sdr['gpa_change'] > 0.5, 'gpa_trend'] = 'improving'
sdr.loc[sdr['gpa_change'] < -0.5, 'gpa_trend'] = 'declining'
sdr

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,missed_deadlines,stress_level,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout,pass_rate,gpa_change,gpa_trend
0,1,23,Male,Brazil,0,0,Low,1,0,1,...,3,8,0,6,37,5.7,0,1.000000,-0.89,declining
1,2,18,Other,USA,1,1,High,1,1,0,...,1,7,0,6,15,5.4,1,0.785714,-0.03,stable
2,3,24,Other,USA,0,0,High,0,1,1,...,0,2,0,3,4,4.3,0,0.823529,-0.70,declining
3,4,25,Female,USA,0,1,Middle,1,1,0,...,1,5,0,7,10,6.7,0,0.846154,-0.41,stable
4,5,15,Female,Canada,0,0,Low,1,1,0,...,0,7,0,10,24,5.8,0,0.875000,-1.05,declining
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,99996,25,Female,USA,0,0,Middle,1,1,1,...,0,4,0,6,11,5.9,0,1.000000,-0.98,declining
99996,99997,24,Female,India,0,1,Middle,0,1,1,...,3,7,1,5,21,5.5,1,0.500000,0.21,stable
99997,99998,27,Female,USA,0,0,Middle,1,1,1,...,2,2,0,6,29,5.8,0,1.000000,-0.57,declining
99998,99999,21,Male,UK,0,0,High,0,1,0,...,2,4,1,10,43,6.2,0,0.714286,-0.99,declining


4 Verify that, for every student, units_passed + units_failed equals units_enrolled; if not, report the rows violating this condition

In [6]:
sdr['units_sum'] = sdr['units_passed'] +sdr['units_failed']
sdr['units_equals'] = sdr['units_sum'] == sdr['units_enrolled']
error = sdr[sdr['units_equals'] == False]
error

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,mental_health_support,satisfaction_score,commute_time_mins,sleep_hours,dropout,pass_rate,gpa_change,gpa_trend,units_sum,units_equals


5 Compute the dropout rate for each value of gpa_trend, and the person_id of the 10 students with the largest GPA drop

In [7]:
dr_rate_t = sdr.groupby('gpa_trend')['dropout'].mean().reset_index()
dr_rate_t.columns = ['gpa_trend', 'dropout_rate']
dr_rate_t

,gpa_trend,dropout_rate
0,declining,0.265086
1,improving,0.589158
2,stable,0.391231


In [8]:
sdr_sorted = sdr.sort_values('gpa_change')
top10_d = sdr_sorted[['person_id', 'gpa_change']].head(10)
top10_d

,person_id,gpa_change
23225,23226,-2.49
96224,96225,-2.40
88489,88490,-2.37
51824,51825,-2.35
58935,58936,-2.33
46714,46715,-2.32
43035,43036,-2.30
72663,72664,-2.27
76705,76706,-2.23
23427,23428,-2.23


6 Partition the students into the following groups, according to weekly_work_hours: 0, 1-10, 11-20, 21-30, more than 30 (all intervals include both extremes)

In [9]:
sdr['work_g'] = 'more than 30'
sdr.loc[sdr['weekly_work_hours'] == 0, 'work_g'] = '0'
sdr.loc[(sdr['weekly_work_hours'] >= 1) & (sdr['weekly_work_hours'] <= 10), 'work_g'] = '1-10'
sdr.loc[(sdr['weekly_work_hours'] >= 11) & (sdr['weekly_work_hours'] <= 20), 'work_g'] = '11-20'
sdr.loc[(sdr['weekly_work_hours'] >= 21) & (sdr['weekly_work_hours'] <= 30), 'work_g'] = '21-30'
sdr

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,satisfaction_score,commute_time_mins,sleep_hours,dropout,pass_rate,gpa_change,gpa_trend,units_sum,units_equals,work_g
0,1,23,Male,Brazil,0,0,Low,1,0,1,...,6,37,5.7,0,1.000000,-0.89,declining,11,True,21-30
1,2,18,Other,USA,1,1,High,1,1,0,...,6,15,5.4,1,0.785714,-0.03,stable,14,True,0
2,3,24,Other,USA,0,0,High,0,1,1,...,3,4,4.3,0,0.823529,-0.70,declining,17,True,11-20
3,4,25,Female,USA,0,1,Middle,1,1,0,...,7,10,6.7,0,0.846154,-0.41,stable,13,True,0
4,5,15,Female,Canada,0,0,Low,1,1,0,...,10,24,5.8,0,0.875000,-1.05,declining,8,True,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,99996,25,Female,USA,0,0,Middle,1,1,1,...,6,11,5.9,0,1.000000,-0.98,declining,6,True,11-20
99996,99997,24,Female,India,0,1,Middle,0,1,1,...,5,21,5.5,1,0.500000,0.21,stable,12,True,11-20
99997,99998,27,Female,USA,0,0,Middle,1,1,1,...,6,29,5.8,0,1.000000,-0.57,declining,11,True,21-30
99998,99999,21,Male,UK,0,0,High,0,1,0,...,10,43,6.2,0,0.714286,-0.99,declining,7,True,0


7 Build a risk score by summing:


2 points if the tuition is not up to date;

1 point if stress_level is at least 8;

1 point if sleep_hours is smaller than 5;

1 point if the student has a part-time job with more
than 20 weekly work hours;

2 points if attendance_rate is smaller than 60;

1 point if missed_deadlines is larger than 3.

For each value of the risk score, compute the number of students and the dropout rate

In [10]:
sdr['p_tuition'] = (sdr['tuition_up_to_date'] == False) * 2
sdr['p_stress'] = (sdr['stress_level'] >= 8)
sdr['p_sleep'] = (sdr['sleep_hours'] < 5)
sdr['p_part-time'] = (sdr['weekly_work_hours'] > 20)
sdr['p_attendance'] = (sdr['attendance_rate'] < 60) * 2
sdr['p_deadlines'] = (sdr['missed_deadlines'] > 3)
sdr['risk_score'] = (sdr['p_tuition'] + sdr['p_stress'] + sdr['p_sleep'] + sdr['p_part-time'] + sdr['p_attendance'] + sdr['p_deadlines'])
score_pe = sdr.groupby('risk_score')['person_id'].count().reset_index()
score_pe.columns = ['risk_score', 'students']
score_dr = sdr.groupby('risk_score')['dropout'].mean().reset_index()
risk_sum = pd.merge(score_pe, score_dr, on = 'risk_score')
risk_sum

,risk_score,students,dropout
0,0,26955,0.152847
1,1,24115,0.252084
2,2,21984,0.337200
3,3,14557,0.452703
4,4,7613,0.577433
5,5,3320,0.673494
6,6,1149,0.794604
7,7,267,0.827715
8,8,40,0.900000


8 Read the GDP per capita dataset published by Our World in Data directly from the URL

https://ourworldindata.org/grapher/gdp-per-capita-worldbank.csv?v=1&csvType=full&useColumnShortNames=true
and store it in a dataframe called gdp. The column ny_gdp_pcap_pp_kd contains the GDP per capita.

Clean the dataset, removing the rows that do not correspond to a single country (e.g. World, High-income countries). For each country keep only the row with the most recent year

In [11]:
gdp = pd.read_csv('gdp-per-capita-worldbank.csv')
gdp

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
0,Afghanistan,AFG,2000,1617.8264,Asia
1,Afghanistan,AFG,2001,1454.1108,Asia
2,Afghanistan,AFG,2002,1774.3087,Asia
3,Afghanistan,AFG,2003,1815.9282,Asia
4,Afghanistan,AFG,2004,1776.9182,Asia
...,...,...,...,...,...
7440,Zimbabwe,ZWE,2021,4827.1094,Africa
7441,Zimbabwe,ZWE,2022,5036.7830,Africa
7442,Zimbabwe,ZWE,2023,5218.0454,Africa
7443,Zimbabwe,ZWE,2024,5211.8975,Africa


In [12]:
gdp_countries = gdp.dropna(subset=["owid_region"])
year = gdp_countries.groupby('entity')['year'].max().reset_index()
gdp_clean = pd.merge(gdp_countries, year, on = ['entity', 'year'])
gdp_clean

,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
0,Afghanistan,AFG,2024,1964.4523,Asia
1,Albania,ALB,2025,22724.2930,Europe
2,Algeria,DZA,2025,15882.7140,Africa
3,Andorra,AND,2025,67993.0550,Europe
4,Angola,AGO,2025,8760.1900,Africa
...,...,...,...,...,...
194,Uzbekistan,UZB,2025,11622.8550,Asia
195,Vanuatu,VUT,2025,3866.9980,Oceania
196,Vietnam,VNM,2025,15457.3500,Asia
197,Zambia,ZMB,2025,3742.2563,Africa


9 Join the GPD data with the students data. Compute the dropout rate for each continent

In [13]:
sdr_countries = list(sdr['nationality'].dropna().unique())
gdp_countries = list(gdp_clean['entity'].dropna().unique())
country = [c for c in sdr_countries if c not in gdp_countries]
country

['USA', 'UK']

In [14]:
country_name = {'UK' : 'United Kingdom', 'USA' : 'United States'}
sdr['country'] = sdr['nationality'].replace(country_name)
total_dt = pd.merge(sdr, gdp_clean, left_on = 'country', right_on = 'entity', how = 'left')
total_dt

,person_id,age,gender,nationality,first_generation,disability_status,family_income_level,scholarship_holder,tuition_up_to_date,part_time_job,...,p_part-time,p_attendance,p_deadlines,risk_score,country,entity,code,year,ny_gdp_pcap_pp_kd,owid_region
0,1,23,Male,Brazil,0,0,Low,1,0,1,...,True,0,False,4,Brazil,Brazil,BRA,2025,20024.697,South America
1,2,18,Other,USA,1,1,High,1,1,0,...,False,2,False,2,United States,United States,USA,2025,76931.310,North America
2,3,24,Other,USA,0,0,High,0,1,1,...,False,0,False,1,United States,United States,USA,2025,76931.310,North America
3,4,25,Female,USA,0,1,Middle,1,1,0,...,False,0,False,0,United States,United States,USA,2025,76931.310,North America
4,5,15,Female,Canada,0,0,Low,1,1,0,...,False,0,False,0,Canada,Canada,CAN,2025,57989.580,North America
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
99995,99996,25,Female,USA,0,0,Middle,1,1,1,...,False,0,False,0,United States,United States,USA,2025,76931.310,North America
99996,99997,24,Female,India,0,1,Middle,0,1,1,...,False,0,False,0,India,India,IND,2025,10039.071,Asia
99997,99998,27,Female,USA,0,0,Middle,1,1,1,...,True,2,False,3,United States,United States,USA,2025,76931.310,North America
99998,99999,21,Male,UK,0,0,High,0,1,0,...,False,0,False,0,United Kingdom,United Kingdom,GBR,2025,53993.117,Europe


In [15]:
dr_c = total_dt.groupby('owid_region')['dropout'].mean().reset_index()
dr_c

,owid_region,dropout
0,Africa,0.316721
1,Asia,0.322908
2,Europe,0.322762
3,North America,0.317679
4,Oceania,0.321981
5,South America,0.317620
